In [4]:
#Menjalankan SparkSession
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TugasMandiri4") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

26/09/16 22:20:08 WARN Utils: Your hostname, nada resolves to a loopback address: 127.0.1.1; using 192.168.1.4 instead (on interface wlp1s0)
26/09/16 22:20:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 22:20:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
#A. Membaca dataset dari HDFS, tampilkan printSchema(), count(), dan 10 baris pertama show()
df = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv",
    header=True, inferSchema=True
)

df.printSchema()
print("jumlah baris: ", df.count())
df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

jumlah baris:  1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|       

In [32]:
#B. Menangani data kosong
from pyspark.sql.functions import col

#Banyaknya nilai kosong pada kolom rating
jumlah_kosong = df.filter(col("rating").isNull()).count()
print("Jumlah nilai kosong pada rating:", jumlah_kosong)

Jumlah nilai kosong pada rating: 204


In [33]:
#Rata-rata rating 
df.select("rating").summary("mean").show()

+-------+------------------+
|summary|            rating|
+-------+------------------+
|   mean|4.1457286432160805|
+-------+------------------+



In [34]:
#Mengisi nilai kosong
df = df.na.fill({"rating": 4.1457286432160805})

#Mengecek kembali jumlah nilai kosong
df.filter(col("rating").isNull()).count()

0

In [29]:
#Alasan menggunakan na.fill({})
"Karena sebelumnya ada 204 nilai yang kosong pada kolom rating. Yang kemudian ditangani menggunakan na.fill({}) dengan diisi nilai rata-rata rating. Metode ini dipilih untuk tetap mempertahankan semua data yang ada di dalam dataset, agar 204 data yang ratingnya kosong tidak harus dihapus."

'Karena sebelumnya ada 204 nilai yang kosong pada kolom rating. Yang kemudian ditangani menggunakan na.fill({}) dengan diisi nilai rata-rata rating. Metode ini dipilih untuk tetap mempertahankan semua data yang ada di dalam dataset, agar 204 data yang ratingnya kosong tidak harus dihapus.'

In [39]:
#C. Transformasi data
from pyspark.sql.functions import col

#MEenambahkan kolom unit_terjual * harga_satuan
df = df.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga_satuan")
)

df.select("unit_terjual", "harga_satuan", "total_pendapatan").show(10)

+------------+------------+----------------+
|unit_terjual|harga_satuan|total_pendapatan|
+------------+------------+----------------+
|           3|       90000|          270000|
|           3|      200000|          600000|
|           8|       60000|          480000|
|           6|      350000|         2100000|
|          10|       60000|          600000|
|           5|       20000|          100000|
|           2|       20000|           40000|
|           8|       90000|          720000|
|           7|       20000|          140000|
|          10|       90000|          900000|
+------------+------------+----------------+
only showing top 10 rows



In [41]:
#Kolom tier_transaksi
from pyspark.sql.functions import when

df = df.withColumn(
    "tier_transaksi", 
    when(col("total_pendapatan") > 500000, "Besar")
    .otherwise("Kecil")
)

df.select(
    "unit_terjual",
    "harga_satuan",
    "total_pendapatan",
    "tier_transaksi"
).show(10)

+------------+------------+----------------+--------------+
|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+------------+------------+----------------+--------------+
|           3|       90000|          270000|         Kecil|
|           3|      200000|          600000|         Besar|
|           8|       60000|          480000|         Kecil|
|           6|      350000|         2100000|         Besar|
|          10|       60000|          600000|         Besar|
|           5|       20000|          100000|         Kecil|
|           2|       20000|           40000|         Kecil|
|           8|       90000|          720000|         Besar|
|           7|       20000|          140000|         Kecil|
|          10|       90000|          900000|         Besar|
+------------+------------+----------------+--------------+
only showing top 10 rows



In [48]:
#D. Analisis dengan GroupBy
from pyspark.sql.functions import sum, desc, avg

#1. Kategori dengan total_pendapatan tertinggi
hasil_kategori = df.groupBy("kategori") \
    .sum("total_pendapatan") \
    .withColumnRenamed("sum(total_pendapatan)", "total_pendapatan") \
    .orderBy(desc("total_pendapatan"))

hasil_kategori.show()

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|        Rumah Tangga|       138665000|
|   Makanan & Minuman|       131890000|
|Kesehatan & Kecan...|       128595000|
|            Olahraga|       126650000|
|             Fashion|       124075000|
|          Elektronik|       110295000|
+--------------------+----------------+



In [47]:
#2. Kota dengan tier_transaksi "Besar" terbanyak
kota = df.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .count() \
    .orderBy(desc("count"))

kota.show()

+----------+-----+
|      kota|count|
+----------+-----+
|      Solo|   92|
|  Magelang|   78|
|   Kebumen|   78|
|Yogyakarta|   75|
| Purworejo|   66|
|  Semarang|   65|
+----------+-----+



In [51]:
#3. Rata-rata rating untuk masing-masing metode_pembayaran
rating_mp = df.groupBy("metode_pembayaran") \
    .agg(avg("rating").alias ("rata_rata_rating")) \
    .orderBy(desc("rata_rata_rating"))

rating_mp.show()

+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD| 4.167310656870009|
|    Transfer Bank|   4.1592349097265|
|         E-Wallet| 4.137728643216084|
|     Kartu Kredit|4.1179474608816475|
+-----------------+------------------+



In [8]:
#E. Menyimpan hasil olahan bagian C ke HDFS

df.write \
  .mode("overwrite") \
  .option("header", True) \
  .csv("hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026_output")

In [9]:
#Verifikasi
df_hasil = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026_output",
    header=True,
    inferSchema=True
)
df_hasil.show(5)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|
|ORD-3004|2026-09-10 00:00:00|        Rumah Tangga|Yogyakarta|          10|       60000|         E-Wallet|   4.0|
+--------+-------------------+--------------------+----------+------------+------------+

In [10]:
#Alasan Spark menyimpan hasil dalam bbrp berkas partisi
" Karena spark bekerja secara terdistribusi (membagi data menjadi beberapa bagian kecil. Saat proses penyimpanan, tiap worker (executor akan menulis bagian datanya masing-masing ke HDFS secara bersamaan. Tujuannya yaitu agar mempercepat proses penyimpanan data dan juga agar tidak terjadi crash."

' Karena spark bekerja secara terdistribusi (membagi data menjadi beberapa bagian kecil. Saat proses penyimpanan, tiap worker (eksekutor) akan menulis bagian datanya masing-masing ke HDFS secara bersamaan. Tujuannya yaitu agar mempercepat proses penyimpanan data dan juga agar tidak terjadi crash.'